# FemurFix — region fine-tuning (5-class) @ 960 px

**SKELETON** — run only after `ml/data/region/` exists (built by
`ml/scripts/prepare_region_dataset.py` from a licensed region-labelled source, e.g. the
Roboflow `proximal-femur-fracture` export) and after clinical review of `review_manifest.csv`.

Classes: `femoral_neck`, `intertrochanteric`, `subtrochanteric`, `femoral_shaft`, `distal_femur`.

Plan (from `ml/README.md`):
- fine-tune from the current checkpoint (`best.pt` of v2 if it beat v1, else the v1 best) at
  `imgsz=960`
- split by patient (the prep script groups filenames by prefix; verify before training)
- hold out a hip-only test set and report **per-class AP** + confusion matrix


In [ ]:
import torch
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
%pip install -q "ultralytics>=8.3,<9" "albumentations>=1.4.21,<2" "onnx>=1.16" "onnxslim>=0.1.31"


In [ ]:
from pathlib import Path
from ultralytics import YOLO

# The dataset was uploaded as birwatkar/femurfix-region-dataset
DATA_YAML = Path("/kaggle/input/birwatkar/femurfix-region-dataset/data.yaml")
assert DATA_YAML.is_file(), "attach birwatkar/femurfix-region-dataset in the kernel's Input panel"

# fine-tune from the v3 backbone (best.pt is the 5-class region model target; v3 is the best
# fracture detector). Attach best_v3.pt from Kaggle as a second dataset if available.
model = YOLO("yolo11s.pt")  # or YOLO("/kaggle/input/.../best_v3.pt") if attached
model.train(
    data=str(DATA_YAML),
    epochs=60,
    patience=20,
    imgsz=960,
    batch=8,
    device=0,
    seed=0,
    deterministic=True,
    cache=True,
    fliplr=0.5,
    degrees=10.0,
    close_mosaic=12,
    project="/kaggle/working/runs",
    name="region_yolo11s_960",
)


In [ ]:
import json
from pathlib import Path
from ultralytics import YOLO

run_dir = Path("/kaggle/working/runs/region_yolo11s_960")
best = run_dir / "weights" / "best.pt"
assert best.is_file()
metrics = YOLO(str(best)).val(data=str(DATA_YAML), split="test", imgsz=960, batch=8, device=0, plots=True)
summary = {
    "per_class_ap50": {k: float(v) for k, v in metrics.box.maps.items()},
    "mAP50": float(metrics.box.map50),
    "mAP50-95": float(metrics.box.map),
}
Path("/kaggle/working/region_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))


In [ ]:
from pathlib import Path
from ultralytics import YOLO

best = Path("/kaggle/working/runs/region_yolo11s_960/weights/best.pt")
onnx_path = Path(YOLO(str(best)).export(format="onnx", imgsz=960, opset=12, simplify=True, dynamic=False))
target = Path("/kaggle/working/femur_region.onnx")
target.write_bytes(onnx_path.read_bytes())
print("download-ready:", target, target.stat().st_size, "bytes")


## Outputs
- `/kaggle/working/femur_region.onnx` — region detector candidate
- `/kaggle/working/region_summary.json` — per-class AP on the held-out set

Integration requires the backend changes in `ml_detector.py` (region class names read from ONNX
metadata already work — see `_read_names`) plus `xray_analyzer.py` to consume the class of the
top-scoring box instead of the geometric region fallback. Do **not** ship a region model fine-tuned
solely on non-femur anatomy.
